In [4]:
import re
import numpy as np
from collections import defaultdict

text = """
Natural language processing enables computers to understand human language.
Language models learn word relationships and semantic meanings.
"""

embedding_dim = 5
window_size = 2
epochs = 100
learning_rate = 0.01

print("\nSTEP 1: Preprocessing Text\n")

text = text.lower()
text = re.sub(r'[^a-zA-Z\s]', '', text)
words = text.split()

print("Words:", words)

print("\nSTEP 2: Building Vocabulary\n")

vocab = list(set(words))
word_to_index = {word: i for i, word in enumerate(vocab)}
index_to_word = {i: word for word, i in word_to_index.items()}
V = len(vocab)

print("Vocabulary:", vocab)
print("Vocabulary Size:", V)

print("\nSTEP 3: Generating Training Data\n")

training_data = []

for i, word in enumerate(words):
    target = word_to_index[word]

    for j in range(-window_size, window_size + 1):
        if j != 0 and 0 <= i + j < len(words):
            context = word_to_index[words[i + j]]
            training_data.append((target, context))

print("Sample Training Pairs:", training_data[:10])

print("\nSTEP 4: Initializing Weights\n")

W1 = np.random.rand(V, embedding_dim)
W2 = np.random.rand(embedding_dim, V)

print("\nSTEP 5: Training Model\n")

def softmax(x):
    e_x = np.exp(x - np.max(x))
    return e_x / e_x.sum()

for epoch in range(epochs):
    loss = 0

    for target, context in training_data:
        x = np.zeros(V)
        x[target] = 1

        h = np.dot(W1.T, x)
        u = np.dot(W2.T, h)
        y_pred = softmax(u)

        loss -= np.log(y_pred[context] + 1e-9)

        e = y_pred.copy()
        e[context] -= 1

        W2 -= learning_rate * np.outer(h, e)
        W1 -= learning_rate * np.outer(x, np.dot(W2, e))

    if epoch % 20 == 0:
        print(f"Epoch {epoch}, Loss: {loss:.4f}")

print("\nSTEP 6: Word Embeddings\n")

embeddings = W1

for word in vocab:
    print(f"{word} → {embeddings[word_to_index[word]]}")

print("\nSTEP 7: Cosine Similarity\n")

def cosine_similarity(vec1, vec2):
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))

word1 = "language"
word2 = "models"

vec1 = embeddings[word_to_index[word1]]
vec2 = embeddings[word_to_index[word2]]

similarity = cosine_similarity(vec1, vec2)

print(f"Similarity between '{word1}' and '{word2}': {similarity:.4f}")



STEP 1: Preprocessing Text

Words: ['natural', 'language', 'processing', 'enables', 'computers', 'to', 'understand', 'human', 'language', 'language', 'models', 'learn', 'word', 'relationships', 'and', 'semantic', 'meanings']

STEP 2: Building Vocabulary

Vocabulary: ['meanings', 'understand', 'models', 'relationships', 'computers', 'processing', 'word', 'learn', 'semantic', 'human', 'enables', 'language', 'and', 'natural', 'to']
Vocabulary Size: 15

STEP 3: Generating Training Data

Sample Training Pairs: [(13, 11), (13, 5), (11, 13), (11, 5), (11, 10), (5, 13), (5, 11), (5, 10), (5, 4), (10, 11)]

STEP 4: Initializing Weights


STEP 5: Training Model

Epoch 0, Loss: 173.3539
Epoch 20, Loss: 160.1396
Epoch 40, Loss: 150.0575
Epoch 60, Loss: 138.4426
Epoch 80, Loss: 127.6600

STEP 6: Word Embeddings

meanings → [ 1.03718478  0.47602324  0.64820208 -0.82386495  1.11666336]
understand → [-0.06619212  0.13351603  1.25842641  1.45825615 -0.13294429]
models → [0.84930682 0.32860448 1.245732